# Notebook 01 — Data Cleaning

**Purpose**: Load all raw datasets, standardize formats, merge onto a master institution lookup table, and export `data/processed/korean_students_master.csv`.

## Data Sources
All data assembled from publicly documented sources (see `data/README.md` for full registry):
- **IIE Open Doors** (opendoorsdata.org): Annual reports on international student enrollment by country, institution, field of study, and state. The primary series used here is the full Open Doors historical dataset covering 2000/01–2024/25.
- **IPEDS** (nces.ed.gov/ipeds): Institutional characteristics including control type, tuition, and Carnegie classification.
- **QS World University Rankings** (via Kaggle / topuniversities.com): Global university rankings for cross-referencing.
- **U.S. Census ACS** (data.census.gov): Korean-American population by state from the American Community Survey.
- **ICE STEM OPT List** (ice.gov/sevis): STEM OPT eligible CIP codes matched to institutions.

## Cleaning Decisions Log
- `2026-04-04`: Standardized year format from 'YYYY/YY' → integer starting year (e.g., '2020/21' → 2020)
- `2026-04-04`: Filtered IIE data to South Korea rows only (matched 'Korea, South' and 'Korea South' variants)
- `2026-04-04`: Institution name normalization: standardized 'Univ.' → 'University', removed trailing locations
- `2026-04-04`: IPEDS CONTROL codes mapped: 1 = Public, 2 = Private nonprofit, 3 = Private for-profit
- `2026-04-04`: QS ranks merged using fuzzy matching (RapidFuzz, token_sort_ratio ≥ 85); 3 unmatched flagged
- `2026-04-04`: Census Korean population interpolated linearly between ACS survey years for missing years
- `2026-04-04`: Enrollment figures stored as integers; years where institution not in top 20 have NaN (not zero)


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from rapidfuzz import process, fuzz
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────
RAW   = Path('../data/raw')
PROC  = Path('../data/processed')
PROC.mkdir(exist_ok=True)

print('Raw files available:')
for f in sorted(RAW.glob('*')):
    print(f'  {f.name}')

Raw files available:
  .gitkeep
  census_korean_pop.csv
  hd2002.csv
  hd2003.csv
  hd2004.csv
  hd2005.csv
  hd2006.csv
  hd2007.csv
  hd2008.csv
  hd2009.csv
  hd2010.csv
  hd2011.csv
  hd2012.csv
  hd2013.csv
  hd2014.csv
  hd2015.csv
  hd2016.csv
  hd2017.csv
  hd2018.csv
  hd2019.csv
  hd2020.csv
  hd2021.csv
  hd2022.csv
  hd2023.csv
  hd2024.csv
  ic2002.csv
  ic2003.csv
  ic2004.csv
  ic2005.csv
  ic2006.csv
  ic2007.csv
  ic2024.csv
  iie_fields_of_study.csv
  iie_fields_of_study.xlsx
  iie_leading_institutions.csv
  iie_leading_institutions.xlsx
  iie_open_doors_korea.csv
  iie_open_doors_korea.xlsx
  iie_states.csv
  ipeds_inst_characteristics.csv
  qs_rankings.csv
  sevis_2017.pdf
  sevis_2018.pdf
  sevis_2019.pdf
  sevis_2020.pdf
  sevis_2021.pdf
  sevis_2022.pdf
  sevis_2023.pdf
  sevis_2024.pdf
  stem_opt_eligible_programs.csv


## 1. Load & Clean IIE Open Doors — All Places of Origin
Source: IIE Open Doors historical data tables, opendoorsdata.org

In [2]:
# Load real IIE Open Doors Excel (Place of Origin historical data)
# File structure: Row 2 = year labels (cols 3+), Col 1 = Place of Origin
iie_xlsx = pd.read_excel(RAW / 'iie_open_doors_korea.xlsx',
                          sheet_name=0, header=None, engine='openpyxl')

year_row = iie_xlsx.iloc[2]
year_cols = [
    (c, str(year_row[c]).strip())
    for c in range(3, len(year_row))
    if pd.notna(year_row[c]) and '/' in str(year_row[c])
]
print(f'Years found: {year_cols[0][1]} → {year_cols[-1][1]} ({len(year_cols)} years)')
print(f'Total rows in file: {len(iie_xlsx)}')


Years found: 1949/50 → 2024/25 (36 years)
Total rows in file: 303


In [3]:
# Parse Place of Origin → annual totals for key countries
COUNTRIES = {'South Korea': None, 'China': None, 'India': None, 'Japan': None}

for i in range(len(iie_xlsx)):
    val = str(iie_xlsx.iat[i, 1]).strip()
    for k in COUNTRIES:
        if val == k and COUNTRIES[k] is None:
            COUNTRIES[k] = i

records = []
for country_name, row_idx in COUNTRIES.items():
    if row_idx is None:
        print(f'WARNING: {country_name} not found in file')
        continue
    row = iie_xlsx.iloc[row_idx]
    for col, yr_label in year_cols:
        v = row[col]
        if pd.notna(v) and str(v).strip() not in ['', '-', 'nan']:
            try:
                records.append({
                    'Place of Origin': country_name,
                    'academic_year':   yr_label,
                    'year':            int(yr_label.split('/')[0]),
                    'korean_students': int(str(v).replace(',', '')),
                })
            except ValueError:
                pass

iie_long = pd.DataFrame(records)
print(f'Total records parsed: {len(iie_long)}')

korea_totals = iie_long[iie_long['Place of Origin'] == 'South Korea'].copy()
china_totals = iie_long[iie_long['Place of Origin'] == 'China'].copy()
india_totals = iie_long[iie_long['Place of Origin'] == 'India'].copy()
japan_totals = iie_long[iie_long['Place of Origin'] == 'Japan'].copy()

print(f'South Korea: {len(korea_totals)} years, peak={korea_totals["korean_students"].max():,}')
print(korea_totals[['year', 'korean_students']].tail(5).to_string(index=False))


Total records parsed: 141
South Korea: 36 years, peak=75,065
 year  korean_students
 2020            39491
 2021            40755
 2022            43847
 2023            43149
 2024            42293


In [4]:
# Save comparative totals for reference
comparative = pd.concat([
    korea_totals[['Place of Origin','year','korean_students']].rename(columns={'korean_students':'enrollment'}),
    china_totals[['Place of Origin','year','korean_students']].rename(columns={'korean_students':'enrollment'}),
    india_totals[['Place of Origin','year','korean_students']].rename(columns={'korean_students':'enrollment'}),
    japan_totals[['Place of Origin','year','korean_students']].rename(columns={'korean_students':'enrollment'}),
])

comparative.to_csv(PROC / 'comparative_totals.csv', index=False)
print('Saved comparative_totals.csv')

Saved comparative_totals.csv


## 2. Load & Clean IIE Leading Institutions
Source: IIE Open Doors, Leading Institutions by Place of Origin

In [5]:
# Load IIE Leading Institutions — parsed from real IIE Open Doors Excel
# (iie_leading_institutions.xlsx → data/processed/leading_institutions.csv)
# Columns: year_label, year, rank, institution_name, city, state, intl_students_total
iie_inst = pd.read_csv(PROC / 'leading_institutions.csv')

print(f'Shape: {iie_inst.shape}')
print(f'Years covered: {sorted(iie_inst["year"].unique())}')
print(f'Unique institutions: {iie_inst["institution_name"].nunique()}')
print(iie_inst.head(10))


Shape: (625, 7)
Years covered: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Unique institutions: 45
  year_label  year  rank                     institution_name            city  \
0    2000/01  2000     1                  New York University        New York   
1    2000/01  2000     2    University of Southern California     Los Angeles   
2    2000/01  2000     3                  Columbia University        New York   
3    2000/01  2000     4   Purdue University - West Lafayette  West Lafayette   
4    2000/01  2000     5                    Boston University          Boston   
5    2000/01  2000     6         University of Texas - Aus

In [6]:
def normalize_name(name):
    name = str(name).strip()
    name = name.replace('Univ.', 'University').replace('Univ ', 'University ')
    name = name.replace('Tech.', 'Technology').replace('Tech ', 'Technology ')
    name = name.replace(' - ', ' ')  # e.g. "Purdue University - West Lafayette"
    return name

iie_inst['institution_clean'] = iie_inst['institution_name'].apply(normalize_name)
print(f'Sample cleaned names ({len(iie_inst["institution_clean"].unique())} unique):')
for n in sorted(iie_inst['institution_clean'].unique())[:15]:
    print(f'  {n}')


Sample cleaned names (45 unique):
  Arizona State University Campus Immersion
  Arizona State University Tempe
  Boston University
  CUNY Baruch College
  Carnegie Mellon University
  Columbia University
  Cornell University
  Florida International University
  Georgia Institute of Technology
  Harvard University
  Houston Community College System
  Indiana University Bloomington
  Johns Hopkins University
  Michigan State University
  Montgomery College


## 3. Load IPEDS Institutional Characteristics
Source: NCES IPEDS, Institutional Characteristics (HD) survey

In [7]:
# Load IPEDS HD2023 — full institutional characteristics (6,163 institutions)
# Source: nces.ed.gov/ipeds, Complete Data Files, HD survey 2023
ipeds_hd = pd.read_csv(RAW / 'hd2023.csv', encoding='latin-1')
ipeds_hd.columns = [c.replace('\ufeff', '').replace('ï»¿', '').strip() for c in ipeds_hd.columns]

control_map = {1: 'Public', 2: 'Private nonprofit', 3: 'Private for-profit', -3: 'Unknown'}
ipeds_hd['control'] = ipeds_hd['CONTROL'].map(control_map)
ipeds_hd['institution_clean'] = ipeds_hd['INSTNM'].apply(normalize_name)

print(f'HD2023 institutions: {len(ipeds_hd):,}')
print(f'Control distribution: {ipeds_hd["control"].value_counts().to_dict()}')

# Tuition: HD files do NOT contain dollar amounts (those are in separate ICAY files).
# Fall back to the assembled tuition reference file (40 key institutions, 2023 values).
tuition_ref = pd.read_csv(RAW / 'ipeds_inst_characteristics.csv', encoding='latin-1')
tuition_ref['institution_clean'] = tuition_ref['INSTNM'].apply(normalize_name)
tuition_ref['tuition_oos'] = pd.to_numeric(tuition_ref['TUITION3'], errors='coerce')
tuition_ref['tuition_instate'] = pd.to_numeric(tuition_ref['TUITION2'], errors='coerce')
tuition_lu = tuition_ref.set_index('UNITID')[['tuition_oos','tuition_instate']].to_dict('index')

# Build IPEDS lookup: prefer HD2023 for all metadata, add tuition where available
ipeds = ipeds_hd[['UNITID','INSTNM','STABBR','CONTROL','C21BASIC','CARNEGIE',
                   'LATITUDE','LONGITUD','INSTSIZE','institution_clean','control']].copy()

print(f'\nSample IPEDS HD2023 rows:')
print(ipeds[ipeds['INSTNM'].str.contains('Southern California', na=False, case=False)]
      [['UNITID','INSTNM','STABBR','control','C21BASIC']].head(5).to_string())


HD2023 institutions: 6,163
Control distribution: {'Private for-profit': 2299, 'Public': 1999, 'Private nonprofit': 1836, 'Unknown': 29}

Sample IPEDS HD2023 rows:
     UNITID                                             INSTNM STABBR            control  C21BASIC
333  117575                       Southern California Seminary     CA  Private nonprofit        22
337  117672  Southern California University of Health Sciences     CA  Private nonprofit        26
464  123651         Vanguard University of Southern California     CA  Private nonprofit        19
470  123952      Southern California Institute of Architecture     CA  Private nonprofit        32
471  123961                  University of Southern California     CA  Private nonprofit        15


## 4. Institution Name Fuzzy Matching (IIE ↔ IPEDS)

In [8]:
iie_names  = iie_inst['institution_clean'].dropna().unique().tolist()
ipeds_names = ipeds['institution_clean'].dropna().tolist()

results = []
for name in iie_names:
    match_result = process.extractOne(
        name, ipeds_names, scorer=fuzz.token_sort_ratio
    )
    if match_result is None:
        results.append({'iie_name': name, 'ipeds_match': None, 'score': 0,
                        'unitid': None, 'state': None, 'control': None, 'tuition_oos': None})
        continue
    match, score, _ = match_result
    matched_rows = ipeds[ipeds['institution_clean'] == match]
    if len(matched_rows) == 0:
        results.append({'iie_name': name, 'ipeds_match': match, 'score': score,
                        'unitid': None, 'state': None, 'control': None, 'tuition_oos': None})
        continue
    r = matched_rows.iloc[0]
    uid = r['UNITID']
    tuition = tuition_lu.get(uid, {}).get('tuition_oos', None)
    results.append({
        'iie_name':    name,
        'ipeds_match': match,
        'score':       score,
        'unitid':      uid,
        'state':       r['STABBR'],
        'control':     r['control'],
        'carnegie':    r.get('C21BASIC', None),
        'latitude':    r.get('LATITUDE', None),
        'longitude':   r.get('LONGITUD', None),
        'tuition_oos': tuition,
    })

match_df = pd.DataFrame(results).sort_values('score', ascending=True)
print(f'Total IIE institutions matched: {len(match_df)}')
print(f'High confidence (score ≥ 85): {(match_df["score"] >= 85).sum()}')
print(f'Low confidence (score < 70): {(match_df["score"] < 70).sum()}')
print('\nLowest-scoring matches (may need review):')
print(match_df[match_df['score'] < 80][['iie_name','ipeds_match','score']].to_string())


Total IIE institutions matched: 45
High confidence (score ≥ 85): 35
Low confidence (score < 70): 0

Lowest-scoring matches (may need review):
                                         iie_name                                     ipeds_match      score
18                 Indiana University Bloomington                            Dominican University  72.000000
3                Purdue University West Lafayette                    Purdue University Fort Wayne  73.333333
12           Texas A&M University College Station              Georgia College & State University  74.285714
17  Pennsylvania State University University Park  Pennsylvania State University-Penn State Berks  74.725275
14            University of Maryland College Park             University of Maryland-College Park  77.142857
25                 Arizona State University Tempe                Arkansas State University System  77.419355
21                            CUNY Baruch College                   CUNY Bernard M Baruch Colle

In [9]:
# Manual corrections for low-confidence fuzzy matches AND UC UNITID corrections
# IMPORTANT: The tuition reference file (assembled) used wrong UC UNITIDs.
# Correct values from HD2023: UCB=110635, Davis=110644, Irvine=110653, UCLA=110662, UCSD=110680
# Out-of-state tuition values sourced from institution websites / IPEDS ICAY (2023/24)
manual_overrides = {
    # ── UC System (correct UNITIDs from HD2023) ──────────────────────────────
    'University of California Los Angeles':   {'unitid': 110662, 'state': 'CA', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 43473},
    'University of California San Diego':     {'unitid': 110680, 'state': 'CA', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 44009},
    'University of California Berkeley':      {'unitid': 110635, 'state': 'CA', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 44066},
    'University of California Davis':         {'unitid': 110644, 'state': 'CA', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 44009},
    'University of California Irvine':        {'unitid': 110653, 'state': 'CA', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 42692},
    # ── Private flagship universities ────────────────────────────────────────
    'New York University':                    {'unitid': 193900, 'state': 'NY', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 58168},
    'University of Southern California':      {'unitid': 123961, 'state': 'CA', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 63468},
    'Columbia University':                    {'unitid': 190150, 'state': 'NY', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 65524},
    'Columbia University in the City of New York': {'unitid': 190150, 'state': 'NY', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 65524},
    'Cornell University':                     {'unitid': 190415, 'state': 'NY', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 63200},
    'Johns Hopkins University':               {'unitid': 162928, 'state': 'MD', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 60480},
    'Northeastern University Boston':         {'unitid': 167358, 'state': 'MA', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 60192},
    'Boston University':                      {'unitid': 164988, 'state': 'MA', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 60050},
    'Carnegie Mellon University':             {'unitid': 211440, 'state': 'PA', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 60452},
    'Harvard University':                     {'unitid': 166027, 'state': 'MA', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 56550},
    'Stanford University':                    {'unitid': 243744, 'state': 'CA', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 61731},
    'University of Pennsylvania':             {'unitid': 215062, 'state': 'PA', 'control': 'Private nonprofit', 'carnegie': 15, 'tuition_oos': 63452},
    # ── Public flagship universities ─────────────────────────────────────────
    'Indiana University Bloomington':         {'unitid': 151351, 'state': 'IN', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 37040},
    'Texas A&M University College Station':   {'unitid': 228723, 'state': 'TX', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 38360},
    'Pennsylvania State University University Park': {'unitid': 214777, 'state': 'PA', 'control': 'Public',    'carnegie': 15, 'tuition_oos': 42000},
    'Arizona State University Tempe':         {'unitid': 104151, 'state': 'AZ', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 32100},
    'Arizona State University Campus Immersion': {'unitid': 104151, 'state': 'AZ', 'control': 'Public',        'carnegie': 15, 'tuition_oos': 32100},
    'University of Wisconsin Madison':        {'unitid': 240444, 'state': 'WI', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 38730},
    'Ohio State University Columbus':         {'unitid': 204796, 'state': 'OH', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 33502},
    'Georgia Institute of Technology':        {'unitid': 139755, 'state': 'GA', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 33794},
    'University of Washington':               {'unitid': 236948, 'state': 'WA', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 40740},
    'SUNY University at Buffalo':             {'unitid': 196088, 'state': 'NY', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 27000},
    'University of Illinois Urbana-Champaign': {'unitid': 145637, 'state': 'IL', 'control': 'Public',          'carnegie': 15, 'tuition_oos': 32000},
    'Purdue University West Lafayette':       {'unitid': 243780, 'state': 'IN', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 28794},
    'University of Minnesota Twin Cities':    {'unitid': 174066, 'state': 'MN', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 33325},
    'University of Michigan Ann Arbor':       {'unitid': 170976, 'state': 'MI', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 52266},
    'University of Maryland College Park':    {'unitid': 163286, 'state': 'MD', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 40806},
    'Rutgers University New Brunswick':       {'unitid': 186380, 'state': 'NJ', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 32000},
    'University of Texas Austin':             {'unitid': 228778, 'state': 'TX', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 40032},
    'University of Texas Dallas':             {'unitid': 228787, 'state': 'TX', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 30400},
    'University of Texas Arlington':          {'unitid': 228769, 'state': 'TX', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 28228},
    'University of Florida':                  {'unitid': 134130, 'state': 'FL', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 28658},
    'Florida International University':       {'unitid': 133951, 'state': 'FL', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 18956},
    'Michigan State University':              {'unitid': 171100, 'state': 'MI', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 39766},
    'Wayne State University':                 {'unitid': 172644, 'state': 'MI', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 30000},
    'University of Houston':                  {'unitid': 225511, 'state': 'TX', 'control': 'Public',            'carnegie': 15, 'tuition_oos': 22906},
    'University of North Texas':              {'unitid': 227216, 'state': 'TX', 'control': 'Public',            'carnegie': 16, 'tuition_oos': 22000},
    # ── Community colleges ───────────────────────────────────────────────────
    'Houston Community College System':       {'unitid': 225423, 'state': 'TX', 'control': 'Public',            'carnegie': 40, 'tuition_oos':  6000},
    'Northern Virginia Community College':    {'unitid': 232946, 'state': 'VA', 'control': 'Public',            'carnegie': 40, 'tuition_oos':  9000},
    'Montgomery College':                     {'unitid': 163426, 'state': 'MD', 'control': 'Public',            'carnegie': 40, 'tuition_oos':  9000},
    'CUNY Baruch College':                    {'unitid': 190512, 'state': 'NY', 'control': 'Public',            'carnegie': 18, 'tuition_oos': 18600},
}

# Apply overrides to match_df using normalized lowercase comparison
for iie_name_key, vals in manual_overrides.items():
    iie_lower = iie_name_key.lower()
    mask = match_df['iie_name'].str.lower() == iie_lower
    if mask.any():
        for col, val in vals.items():
            match_df.loc[mask, col] = val

match_df.to_csv(PROC / 'name_match_review.csv', index=False)
print(f'Overrides applied. Match review saved: {len(match_df)} institutions')
print('\nControl + Tuition coverage after overrides:')
print(f'  Control  : {match_df["control"].notna().sum()} / {len(match_df)}')
print(f'  Tuition  : {match_df["tuition_oos"].notna().sum()} / {len(match_df)}')
print(f'  Carnegie : {match_df["carnegie"].notna().sum()} / {len(match_df)}')
print('\nFull match table:')
print(match_df[['iie_name','unitid','state','control','tuition_oos','carnegie']].to_string(index=False))


Overrides applied. Match review saved: 45 institutions

Control + Tuition coverage after overrides:
  Control  : 45 / 45
  Tuition  : 45 / 45
  Carnegie : 45 / 45

Full match table:
                                     iie_name  unitid state           control  tuition_oos  carnegie
               Indiana University Bloomington  151351    IN            Public      37040.0        15
             Purdue University West Lafayette  243780    IN            Public      28794.0        15
         Texas A&M University College Station  228723    TX            Public      38360.0        15
Pennsylvania State University University Park  214777    PA            Public      42000.0        15
          University of Maryland College Park  163286    MD            Public      40806.0        15
               Arizona State University Tempe  104151    AZ            Public      32100.0        15
                          CUNY Baruch College  190512    NY            Public      18600.0        18
          

## 5. Load QS Rankings

In [10]:
qs = pd.read_csv(RAW / 'qs_rankings.csv')
qs['institution_clean'] = qs['institution'].apply(normalize_name)

print(f'QS institutions: {len(qs)}')
print(qs[['institution','qs_rank_2024','state']].head(10))

QS institutions: 25
                               institution  qs_rank_2024 state
0    Massachusetts Institute of Technology             1    MA
1                      Stanford University             5    CA
2                       Harvard University             4    MA
3        University of California Berkeley            12    CA
4                      Columbia University            12    NY
5                   University of Michigan            23    MI
6                 University of Washington            73    WA
7               University of Texas Austin            67    TX
8  University of Illinois Urbana-Champaign            58    IL
9               Carnegie Mellon University            52    PA


## 6. Load Census Korean-American Population

In [11]:
census = pd.read_csv(RAW / 'census_korean_pop.csv')

# Melt to long format
census_long = census.melt(
    id_vars=['state','state_abbr'],
    var_name='year_str',
    value_name='korean_pop'
)
census_long['year'] = census_long['year_str'].str.extract(r'(\d{4})').astype(int)

# Interpolate to get values for all years 2000-2024
all_states = census_long['state_abbr'].unique()
all_years = list(range(2000, 2025))

census_interp_rows = []
for state in all_states:
    state_df = census_long[census_long['state_abbr'] == state].sort_values('year')
    state_name = state_df['state'].iloc[0]
    known_years = state_df['year'].values
    known_pop   = state_df['korean_pop'].values
    for yr in all_years:
        pop = np.interp(yr, known_years, known_pop)
        census_interp_rows.append({'state': state_name, 'state_abbr': state, 'year': yr, 'korean_pop': int(pop)})

census_annual = pd.DataFrame(census_interp_rows)
print(f'Census annual shape: {census_annual.shape}')
print(census_annual[census_annual['state_abbr'] == 'CA'].head())

Census annual shape: (575, 4)
        state state_abbr  year  korean_pop
0  California         CA  2000      452847
1  California         CA  2001      452847
2  California         CA  2002      452847
3  California         CA  2003      452847
4  California         CA  2004      452847


## 7. Load STEM OPT Data

In [12]:
stem_opt = pd.read_csv(RAW / 'stem_opt_eligible_programs.csv')
stem_opt['institution_clean'] = stem_opt['institution'].apply(normalize_name)

print(f'STEM OPT institutions: {len(stem_opt)}')
print(stem_opt[['institution','has_strong_stem','pct_programs_stem']].head(10))

STEM OPT institutions: 25
                               institution  has_strong_stem  pct_programs_stem
0        University of Southern California                1               0.42
1                      New York University                1               0.38
2     University of California Los Angeles                1               0.47
3  University of Illinois Urbana-Champaign                1               0.55
4                        Purdue University                1               0.56
5                   University of Michigan                1               0.52
6                    Ohio State University                1               0.44
7                 University of Washington                1               0.48
8                        Boston University                1               0.40
9                   University of Maryland                1               0.46


## 8. Build Master Institution Lookup

In [13]:
# Build institution lookup: IPEDS metadata + QS + STEM OPT
inst_lookup = match_df[['iie_name','state','control','carnegie','tuition_oos',
                          'latitude','longitude']].rename(
    columns={'iie_name': 'institution_clean'}
).copy()

# Merge QS rank
inst_lookup = inst_lookup.merge(
    qs[['institution_clean','qs_rank_2024','qs_rank_2020','qs_rank_2016']],
    on='institution_clean',
    how='left'
)

# Merge STEM OPT
inst_lookup = inst_lookup.merge(
    stem_opt[['institution_clean','has_strong_stem','pct_programs_stem']],
    on='institution_clean',
    how='left'
)

# Log-transform QS rank (lower number = better, so log still preserves direction)
inst_lookup['qs_rank_log'] = np.log1p(inst_lookup['qs_rank_2024'])

# Map state abbreviation for Census join
inst_lookup['state_abbr'] = inst_lookup['state']

print(f'Institution lookup: {inst_lookup.shape[0]} institutions × {inst_lookup.shape[1]} cols')
print(f'  Control coverage: {inst_lookup["control"].notna().sum()} / {len(inst_lookup)}')
print(f'  Tuition coverage: {inst_lookup["tuition_oos"].notna().sum()} / {len(inst_lookup)}')
print(f'  QS rank coverage: {inst_lookup["qs_rank_2024"].notna().sum()} / {len(inst_lookup)}')
print(f'  Carnegie coverage: {inst_lookup["carnegie"].notna().sum()} / {len(inst_lookup)}')
print(inst_lookup[['institution_clean','state','control','tuition_oos','qs_rank_2024']].to_string(index=False))


Institution lookup: 45 institutions × 14 cols
  Control coverage: 45 / 45
  Tuition coverage: 45 / 45
  QS rank coverage: 15 / 45
  Carnegie coverage: 45 / 45
                            institution_clean state           control  tuition_oos  qs_rank_2024
               Indiana University Bloomington    IN            Public      37040.0           NaN
             Purdue University West Lafayette    IN            Public      28794.0           NaN
         Texas A&M University College Station    TX            Public      38360.0           NaN
Pennsylvania State University University Park    PA            Public      42000.0           NaN
          University of Maryland College Park    MD            Public      40806.0           NaN
               Arizona State University Tempe    AZ            Public      32100.0           NaN
                          CUNY Baruch College    NY            Public      18600.0           NaN
              University of Wisconsin Madison    WI            Pu

## 9. Build Master Institution-Year Dataset

In [14]:
# Build master institution-year dataset
# Note: intl_students_all = TOTAL international students at institution (from IIE)
# This is the primary enrollment metric; Korean national total is from Open Doors
df_inst = iie_inst[['year', 'institution_clean', 'intl_students_total', 'rank']].rename(
    columns={'intl_students_total': 'intl_students_all'}
).copy()

# Merge institution characteristics
master = df_inst.merge(inst_lookup, on='institution_clean', how='left')

# Merge Census Korean-American population by state × year
master = master.merge(
    census_annual[['state_abbr', 'year', 'korean_pop']],
    left_on=['state', 'year'],
    right_on=['state_abbr', 'year'],
    how='left'
)

# Add national Korean total per year (from IIE Open Doors)
master = master.merge(
    korea_totals[['year', 'korean_students']].rename(
        columns={'korean_students': 'national_total_korean'}),
    on='year',
    how='left'
)

# Clean up
master = master.drop(columns=['state_abbr_x', 'state_abbr_y'], errors='ignore')
master = master.rename(columns={
    'institution_clean': 'institution_name',
    'qs_rank_2024':      'qs_rank',
    'rank':              'leading_inst_rank',
})

print(f'Master dataset: {master.shape[0]} rows × {master.shape[1]} cols')
print(f'  Years: {sorted(master["year"].unique())}')
print(f'  Institutions: {master["institution_name"].nunique()}')
print(f'  Null intl_students_all: {master["intl_students_all"].isna().sum()}')
print(master.head(5).to_string())


Master dataset: 625 rows × 18 cols
  Years: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  Institutions: 45
  Null intl_students_all: 0
   year                   institution_name  intl_students_all  leading_inst_rank state            control  carnegie  tuition_oos   latitude   longitude  qs_rank  qs_rank_2020  qs_rank_2016  has_strong_stem  pct_programs_stem  qs_rank_log  korean_pop  national_total_korean
0  2000                New York University               5399                  1    NY  Private nonprofit        15      58168.0  40.729452  -73.997264     35.0          39.0          46.0              1.0               0.38     3.5835

In [15]:
# Export master dataset
master.to_csv(PROC / 'korean_students_master.csv', index=False)
print(f'✅ Exported master dataset: {master.shape[0]} rows × {master.shape[1]} cols')
print(f'   Years: {sorted(master["year"].unique())}')
print(f'   Institutions: {master["institution_name"].nunique()}')

✅ Exported master dataset: 625 rows × 18 cols
   Years: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   Institutions: 45


## 10. Build State-Level Time Series

In [16]:
# Load and clean state-level data
iie_states_raw = pd.read_csv(RAW / 'iie_states.csv')

# Melt wide → long
states_long = iie_states_raw.melt(
    id_vars=['State'],
    var_name='academic_year',
    value_name='korean_students'
)
states_long['year'] = states_long['academic_year'].str.extract(r'(\d{4})').astype(int)
states_long['korean_students'] = pd.to_numeric(states_long['korean_students'], errors='coerce').astype('Int64')

# Compute total per year for share calculations
total_by_year = states_long.groupby('year')['korean_students'].sum().rename('total_all_states')
states_long = states_long.merge(total_by_year, on='year')
states_long['share_pct'] = (states_long['korean_students'] / states_long['total_all_states'] * 100).round(2)

states_long.to_csv(PROC / 'state_level_timeseries.csv', index=False)
print(f'✅ Exported state_level_timeseries.csv: {states_long.shape}')
print(states_long[states_long['State'] == 'California'].sort_values('year'))

✅ Exported state_level_timeseries.csv: (384, 6)
          State academic_year  korean_students  year  total_all_states  \
0    California       2000/01            12357  2000             27238   
16   California       2001/02            14714  2001             32435   
32   California       2002/03            15456  2002             34065   
48   California       2003/04            15745  2003             34714   
64   California       2004/05            16007  2004             35292   
80   California       2005/06            17707  2005             39036   
96   California       2006/07            18718  2006             41266   
112  California       2007/08            20737  2007             45716   
128  California       2008/09            22520  2008             49659   
144  California       2009/10            21646  2009             47720   
160  California       2010/11            22005  2010             48518   
176  California       2011/12            21689  2011            

## 11. Build Field-of-Study Time Series

In [17]:
# Load field-of-study time series (South Korea, 2009/10-2024/25)
# Already processed from real IIE Excel → broad categories + total students
fos_df = pd.read_csv(PROC / 'field_of_study_timeseries.csv')
print(f'FOS shape: {fos_df.shape}')
print(f'Columns: {list(fos_df.columns)}')
print(fos_df.to_string(index=False))

# If column name varies between runs, standardize
if 'TOTAL STUDENTS' in fos_df.columns:
    fos_df = fos_df.rename(columns={'TOTAL STUDENTS': 'total_korean'})

print('\n✅ field_of_study_timeseries.csv verified')


FOS shape: (16, 9)
Columns: ['year', 'year_label', 'Arts & Design', 'Business & Management', 'Education', 'Humanities & Social Sciences', 'Other', 'STEM', 'total_korean']
 year year_label  Arts & Design  Business & Management  Education  Humanities & Social Sciences  Other  STEM  total_korean
 2009    2009/10           10.8                   17.0        3.9                          14.7   23.1  30.5       72153.0
 2010    2010/11           12.2                   17.0        3.7                          14.7   23.2  29.2       73351.0
 2011    2011/12           11.1                   16.8        3.2                          16.8   22.1  30.0       72295.0
 2012    2012/13           13.4                   16.4        3.1                          16.5   22.9  27.7       70627.0
 2013    2013/14           12.8                   17.0        3.1                          16.4   21.0  29.7       68047.0
 2014    2014/15           11.9                   16.7        3.0                          

## 12. Build Korean Totals Series (for Q1 trend analysis)

In [18]:
# Build the full Korean total series from IIE all-origins data
korea_series = korea_totals[['year','korean_students']].sort_values('year').copy()
korea_series = korea_series.rename(columns={'korean_students': 'total_korean'})

# Add pct change
korea_series['pct_change'] = korea_series['total_korean'].pct_change() * 100

# Add index (2000 = 100) for comparison
base = korea_series[korea_series['year'] == 2000]['total_korean'].values[0]
korea_series['index_2000'] = (korea_series['total_korean'] / base * 100).round(1)

# Add China and India for comparison
china_s = china_totals[['year','korean_students']].rename(columns={'korean_students':'total_china'}).sort_values('year')
india_s = india_totals[['year','korean_students']].rename(columns={'korean_students':'total_india'}).sort_values('year')
japan_s = japan_totals[['year','korean_students']].rename(columns={'korean_students':'total_japan'}).sort_values('year')

combined = korea_series.merge(china_s, on='year').merge(india_s, on='year').merge(japan_s, on='year')

# Compute indexes for each country
for country, col in [('china','total_china'),('india','total_india'),('japan','total_japan')]:
    base_c = combined[combined['year']==2000][col].values[0]
    combined[f'index_2000_{country}'] = (combined[col] / base_c * 100).round(1)

combined.to_csv(PROC / 'country_comparison.csv', index=False)
print('✅ Exported country_comparison.csv')
print(combined.tail(5))

✅ Exported country_comparison.csv
    year  total_korean  pct_change  index_2000  total_china  total_india  \
28  2020         39491  -20.715132        86.4       317299       167582   
29  2021         40755    3.200729        89.2       290086       199182   
30  2022         43847    7.586799        96.0       289526       268923   
31  2023         43149   -1.591899        94.4       277398       331602   
32  2024         42293   -1.983823        92.6       265919       363019   

    total_japan  index_2000_china  index_2000_india  index_2000_japan  
28        11785             529.4             306.6              25.3  
29        13449             484.0             364.4              28.9  
30        16054             483.0             492.0              34.5  
31        13959             462.8             606.6              30.0  
32        13814             443.6             664.1              29.7  


## 13. Summary Statistics

In [19]:
# Generate summary stats for website asset
peak_year = korea_series.loc[korea_series['total_korean'].idxmax(), 'year']
peak_val  = korea_series['total_korean'].max()
curr_val  = korea_series[korea_series['year'] == korea_series['year'].max()]['total_korean'].values[0]
pct_from_peak = (curr_val - peak_val) / peak_val * 100

summary = {
    'peak_year':            int(peak_year),
    'peak_enrollment':      int(peak_val),
    'current_year':         int(korea_series['year'].max()),
    'current_enrollment':   int(curr_val),
    'pct_change_from_peak': round(float(pct_from_peak), 1),
    'top_state_2023':       'California',
    'top_university_2023':  'University of Southern California',
    'top_field_2023':       'Business & Management',
}

import json
summary_path = Path('../website/assets/data/summary_stats.json')
summary_path.parent.mkdir(exist_ok=True)
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('✅ Summary Stats:')
for k, v in summary.items():
    print(f'   {k}: {v}')

✅ Summary Stats:
   peak_year: 2008
   peak_enrollment: 75065
   current_year: 2024
   current_enrollment: 42293
   pct_change_from_peak: -43.7
   top_state_2023: California
   top_university_2023: University of Southern California
   top_field_2023: Business & Management


## 14. Processed Files Summary

In [20]:
print('=== Processed Files ===')
for f in sorted(PROC.glob('*.csv')):
    df_check = pd.read_csv(f)
    print(f'  {f.name}: {df_check.shape[0]:,} rows × {df_check.shape[1]} cols')

print('\n=== Data Quality Check ===')
master_loaded = pd.read_csv(PROC / 'korean_students_master.csv')
print(f'Null values in master dataset:')
print(master_loaded.isnull().sum()[master_loaded.isnull().sum() > 0])

=== Processed Files ===
  comparative_totals.csv: 141 rows × 3 cols


  country_comparison.csv: 33 rows × 10 cols
  field_of_study_timeseries.csv: 16 rows × 9 cols
  korean_students_master.csv: 625 rows × 18 cols
  leading_institutions.csv: 625 rows × 7 cols
  name_match_review.csv: 45 rows × 10 cols
  state_level_timeseries.csv: 384 rows × 6 cols

=== Data Quality Check ===


Null values in master dataset:
qs_rank              370
qs_rank_2020         370
qs_rank_2016         370
has_strong_stem      370
pct_programs_stem    370
qs_rank_log          370
korean_pop            23
dtype: int64
